- requests 모듈은 HTTP 요청을 보내고 API 응답을 받아오는 역할을 한다.
- xml.etree.ElementTree (ET) 모듈은 API 응답이 XML 형식이므로, 이를 파싱하기 위해 필요하다.

📌 XML을 파싱해야하는 이유
XML 형식을 바로 pandas.DataFrame으로 변환할 수 없기 때문에 먼저 xml.etree.ElementTree를 사용해 데이터를 구조화해야 함.
(만약 API가 JSON으로 제공되면 json 모듈을 사용하면 됨.)

## Data Collection- 국토부 실거래가

In [81]:
import os
import requests
import pandas as pd
import xml.etree.ElementTree as ET

API_URL = "http://apis.data.go.kr/1613000/RTMSDataSvcAptTrade/getRTMSDataSvcAptTrade"
SERVICE_KEY = "jYmQgKsPeM/KffLGjHFgvyTBGB/6ODgIGe2qZjXfsd6NG9d4jMtfl7gmx1MUZHpc65LVhAG3ChnTrqHlBr3AHQ=="

# 법정동 코드 로드
code_file = r"C:\my-project\my-project\아파트실거래가예측\법정동코드 전체자료.txt"
code_df = pd.read_csv(code_file, sep='\t', encoding='utf-8', names=['code', 'name', 'is_exist'])
code_df = code_df[code_df['is_exist'] == '존재']
code_df['code'] = code_df['code'].astype(str)  # 코드 문자열 변환

# 데이터 요청 함수
def get_data(gu_code, base_date):
    params = {"LAWD_CD": gu_code, "DEAL_YMD": base_date, "serviceKey": SERVICE_KEY}
    res = requests.get(API_URL, params=params)
    if res.status_code != 200:
        print(f"API 요청 실패: {res.status_code}")
        return None
    return res

# XML 파싱 함수
def get_items(response):
    if response is None:
        return []
    
    root = ET.fromstring(response.content)
    items = []
    
    for item in root.find('body').find('items'):
        data = {elem.tag.strip(): elem.text.strip() for elem in item.findall('*')}
        items.append(data)
    
    return items

# 특정 구 코드 가져오기
def get_gu_code(gu_name):
    gu_code = code_df[code_df['name'].str.contains(gu_name)]['code'].reset_index(drop=True)
    return str(gu_code[0])[:5] if not gu_code.empty else None

# 저장 경로 설정
SAVE_PATH = r"C:\my-project\my-project\아파트실거래가예측\api_data_collection"

# 연월 리스트 생성 (2015~2023)
years = [f"{y:04d}" for y in range(2015, 2024)]
months = [f"{m:02d}" for m in range(1, 13)]
base_date_list = [f"{y}{m}" for y in years for m in months]

# 데이터 수집 및 저장
def collect_and_save(gu_name):
    gu_code = get_gu_code(gu_name)
    if not gu_code:
        print(f"{gu_name} 코드 없음")
        return
    
    all_items = []
    
    for base_date in base_date_list:
        res = get_data(gu_code, base_date)
        all_items.extend(get_items(res))
    
    if all_items:
        df = pd.DataFrame(all_items)
        file_name = f"{gu_name}_{years[0]}~{years[-1]}.csv"
        full_path = os.path.join(SAVE_PATH, file_name)
        df.to_csv(full_path, index=False, encoding="euc-kr")
        print(f"저장 완료: {full_path}")

# 원하는 구 데이터 수집 및 저장
collect_and_save("동대문구")


저장 완료: C:\my-project\my-project\아파트실거래가예측\api_data_collection\동대문구_2015~2023.csv


In [80]:
import pandas as pd

# Excel 파일 경로
file_path = r"C:\my-project\my-project\아파트실거래가예측\api_data_collection\강남구_2015~2023.csv"

# Excel 파일 읽기
GN_df = pd.read_csv(file_path, encoding='euc-kr')   
GN_df

,aptDong,aptNm,buildYear,buyerGbn,cdealDay,cdealType,dealAmount,dealDay,dealMonth,dealYear,dealingGbn,estateAgentSggNm,excluUseAr,floor,jibun,landLeaseholdGbn,rgstDate,sggCd,slerGbn,umdNm
0,NaN,상아아파트,1981,NaN,NaN,NaN,"153,000",28,1,2015,NaN,NaN,147.740,4,19-4,N,NaN,11680,NaN,삼성동
1,NaN,"현대14차(203,204,205,206동)",1987,NaN,NaN,NaN,"143,000",22,1,2015,NaN,NaN,84.980,5,447,N,NaN,11680,NaN,압구정동
2,NaN,한솔마을,1994,NaN,NaN,NaN,"83,200",27,1,2015,NaN,NaN,84.730,1,731,N,NaN,11680,NaN,일원동
3,NaN,타워팰리스2,2003,NaN,NaN,NaN,"182,000",26,1,2015,NaN,NaN,144.630,29,467-17,N,NaN,11680,NaN,도곡동
4,NaN,상지리츠빌5B,2001,NaN,NaN,NaN,"130,000",26,1,2015,NaN,NaN,152.850,4,127-37,N,NaN,11680,NaN,청담동
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1075,106,래미안대치팰리스,2015,NaN,NaN,NaN,"323,000",30,12,2023,중개거래,서울 강남구,84.980,18,1027,N,24.02.29,11680,NaN,대치동
1076,104,역삼래미안,2005,NaN,NaN,NaN,"182,000",31,12,2023,중개거래,서울 강남구,59.530,17,757,N,24.03.22,11680,NaN,역삼동
1077,804,동익,1993,NaN,NaN,NaN,"142,000",11,12,2023,중개거래,서울 강남구,84.460,14,738,N,24.01.25,11680,NaN,수서동
1078,1단지 110,삼성동힐스테이트 1단지,2008,NaN,NaN,NaN,"380,000",21,12,2023,중개거래,서울 강남구,114.463,9,16-2,N,24.06.11,11680,NaN,삼성동


In [65]:
# dealYear 컬럼과 dealMonth 합쳐서 거래연월 컬럼 생성
GN_df['거래연월'] = GN_df['dealYear'].astype(str) + GN_df['dealMonth'].astype(str).str.zfill(2)

# 결과 출력
print(GN_df[['dealYear', 'dealMonth', '거래연월']].head())

   dealYear  dealMonth    거래연월
0      2015          1  201501
1      2015          1  201501
2      2015          1  201501
3      2015          1  201501
4      2015          1  201501


## Data Collection- 시장 금리 데이터

In [23]:
!pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)


In [75]:
import pandas as pd

# Excel 파일 경로
file_path = r"C:\my-project\my-project\아파트실거래가예측\시장금리_2015-2023.xlsx"

# Excel 파일 읽기 
market_interest_rate_df = pd.read_excel(file_path, index_col=0)

# 데이터 전치 (transpose) - 거래연월을 행으로, 금리종류를 열로 변경
market_interest_rate_df = market_interest_rate_df.T

# 인덱스 명 변경 (거래연월로 거래가 df와 통일일)
market_interest_rate_df.index.name = '거래연월'

# 결과 확인
market_interest_rate_df.head()

,국고채 3년(평균),국고채 5년(평균),국고채 10년(평균),"회사채 3년(AA-, 평균)",CD 91물(평균),"콜금리(1일물,평균)",기준금리
거래연월,,,,,,,
201501월,2.04,2.16,2.42,2.36,2.13,1.99,2.00
201502월,2.02,2.11,2.35,2.30,2.12,1.99,2.00
201503월,1.87,1.97,2.28,2.13,1.95,1.82,1.75
201504월,1.74,1.86,2.18,1.99,1.81,1.73,1.75
201505월,1.88,2.11,2.49,2.12,1.80,1.74,1.75


In [76]:
# 거래연월 값에서 '월' 문자 제거 
market_interest_rate_df.index = market_interest_rate_df.index.str.replace('월', '')

# 결과 확인
market_interest_rate_df.head()

,국고채 3년(평균),국고채 5년(평균),국고채 10년(평균),"회사채 3년(AA-, 평균)",CD 91물(평균),"콜금리(1일물,평균)",기준금리
거래연월,,,,,,,
201501,2.04,2.16,2.42,2.36,2.13,1.99,2.00
201502,2.02,2.11,2.35,2.30,2.12,1.99,2.00
201503,1.87,1.97,2.28,2.13,1.95,1.82,1.75
201504,1.74,1.86,2.18,1.99,1.81,1.73,1.75
201505,1.88,2.11,2.49,2.12,1.80,1.74,1.75


## Data Collection- 데이터 병합

In [77]:
# GN_df와 market_interest_rate_df를 '거래연월' 기준으로 데이터 병합 
merged_df = pd.merge(GN_df, market_interest_rate_df, on='거래연월', how='left')
merged_df.head()

,aptDong,aptNm,buildYear,buyerGbn,cdealDay,cdealType,dealAmount,dealDay,dealMonth,dealYear,...,slerGbn,umdNm,거래연월,국고채 3년(평균),국고채 5년(평균),국고채 10년(평균),"회사채 3년(AA-, 평균)",CD 91물(평균),"콜금리(1일물,평균)",기준금리
0,NaN,상아아파트,1981,NaN,NaN,NaN,"153,000",28,1,2015,...,NaN,삼성동,201501,2.04,2.16,2.42,2.36,2.13,1.99,2.0
1,NaN,"현대14차(203,204,205,206동)",1987,NaN,NaN,NaN,"143,000",22,1,2015,...,NaN,압구정동,201501,2.04,2.16,2.42,2.36,2.13,1.99,2.0
2,NaN,한솔마을,1994,NaN,NaN,NaN,"83,200",27,1,2015,...,NaN,일원동,201501,2.04,2.16,2.42,2.36,2.13,1.99,2.0
3,NaN,타워팰리스2,2003,NaN,NaN,NaN,"182,000",26,1,2015,...,NaN,도곡동,201501,2.04,2.16,2.42,2.36,2.13,1.99,2.0
4,NaN,상지리츠빌5B,2001,NaN,NaN,NaN,"130,000",26,1,2015,...,NaN,청담동,201501,2.04,2.16,2.42,2.36,2.13,1.99,2.0


## 데이터 전처리